# is-differentiable-flag — faded example 1: Wrap a non-differentiable op — fill in the three-gate requires_grad

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `is-differentiable-flag`. The last cell reports your progress on the `Backprop: is_differentiable flag` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: is_differentiable flag` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`is-differentiable-flag`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "is-differentiable-flag"
DD_SUBTOPIC = "Backprop: is_differentiable flag"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When `is_differentiable=False`, the output tensor must always have `requires_grad=False` and `recipe=None`. This is enforced by including `is_differentiable` as the second gate in a three-way AND that computes `requires_grad`.

## Faded exercise 1

Implement `wrap_forward_fn(fwd_fn, is_differentiable=True)`. The function has been partially written for you. Fill in the computation of `requires_grad` so that it gates on all three conditions: the global tracking flag, the per-op flag, and whether any input is tracked.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn, is_differentiable=True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            grad_tracking_enabled
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func


def _test():
    import numpy as np
    from dataclasses import dataclass
    from typing import Any, Optional

    _grad_tracking_enabled = True

    @dataclass
    class _Recipe:
        func: Any
        args: tuple
        kwargs: dict
        parents: dict

    class _MiniTensor:
        def __init__(self, array, requires_grad=False):
            self.array = array
            self.requires_grad = requires_grad
            self.recipe = None

    # Ground truth implementation
    def ref_wrap(fwd_fn, is_differentiable=True):
        def tf(*args, **kwargs):
            raw = tuple(a.array if isinstance(a, _MiniTensor) else a for a in args)
            out_arr = fwd_fn(*raw, **kwargs)
            rg = (_grad_tracking_enabled and is_differentiable
                  and any(isinstance(a, _MiniTensor) and a.requires_grad for a in args))
            out = _MiniTensor(out_arr, rg)
            if rg:
                out.recipe = _Recipe(fwd_fn, raw, kwargs, {i: a for i, a in enumerate(args) if isinstance(a, _MiniTensor)})
            return out
        return tf

    arr = np.array([1.0, 2.0])
    x_tracked = MiniTensor(arr.copy(), requires_grad=True)
    x_frozen = MiniTensor(arr.copy(), requires_grad=False)

    # Non-diff op with tracked input -> False
    non_diff = wrap_forward_fn(np.negative, is_differentiable=False)
    out = non_diff(x_tracked)
    assert out.requires_grad == False, "is_diff=False should force requires_grad=False"
    assert out.recipe is None, "is_diff=False should skip Recipe"

    # Diff op with tracked input -> True
    diff = wrap_forward_fn(np.exp, is_differentiable=True)
    out2 = diff(x_tracked)
    assert out2.requires_grad == True
    assert out2.recipe is not None

    # Diff op with frozen input -> False
    out3 = diff(x_frozen)
    assert out3.requires_grad == False
    assert out3.recipe is None


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
from dataclasses import dataclass
from typing import Any, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn, is_differentiable=True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            grad_tracking_enabled
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```
</details>